# 🛰️ Mission Artemis 2 - Mission Control Dashboard

Real-time monitoring dashboard for Artemis 2 mission.

**Panels:**
- Vehicle Status (altitude, velocity, fuel)
- Crew Vitals (heart rate, O2 for all 4 astronauts)
- Environmental (cabin conditions, radiation)
- Mission Timeline (events and milestones)

In [ ]:
# Configuration
KUSTO_CLUSTER = "<YOUR_EVENTHOUSE_URI>"
DATABASE_NAME = "MissionData"

In [ ]:
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
from azure.kusto.data.helpers import dataframe_from_result_table
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, clear_output
import time

In [ ]:
# Initialize Kusto client
kcsb = KustoConnectionStringBuilder.with_az_cli_authentication(KUSTO_CLUSTER)
client = KustoClient(kcsb)

def run_query(query: str):
    response = client.execute(DATABASE_NAME, query)
    return dataframe_from_result_table(response.primary_results[0])

In [ ]:
# Get latest session
session_query = """
VehicleTelemetry
| summarize LastEvent = max(Timestamp) by SessionId
| top 1 by LastEvent desc
"""
session_df = run_query(session_query)
SESSION_ID = session_df['SessionId'].iloc[0] if len(session_df) > 0 else None
print(f"🚀 Active Mission Session: {SESSION_ID[:8] if SESSION_ID else 'None'}...")

## 🛸 Vehicle Telemetry

In [ ]:
vehicle_query = f"""
VehicleTelemetry
| where SessionId == "{SESSION_ID}"
| order by MissionTime asc
| project MissionTime, Altitude, Velocity, Acceleration, FuelRemaining, Phase
"""
vehicle_df = run_query(vehicle_query)

# Create multi-panel vehicle chart
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Altitude (km)', 'Velocity (km/h)', 'G-Force', 'Fuel (%)')
)

fig.add_trace(go.Scatter(x=vehicle_df['MissionTime'], y=vehicle_df['Altitude'], 
                         mode='lines', name='Altitude', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=vehicle_df['MissionTime'], y=vehicle_df['Velocity'],
                         mode='lines', name='Velocity', line=dict(color='green')), row=1, col=2)
fig.add_trace(go.Scatter(x=vehicle_df['MissionTime'], y=vehicle_df['Acceleration'],
                         mode='lines', name='G-Force', line=dict(color='red')), row=2, col=1)
fig.add_trace(go.Scatter(x=vehicle_df['MissionTime'], y=vehicle_df['FuelRemaining'],
                         mode='lines', name='Fuel', line=dict(color='orange')), row=2, col=2)

fig.update_layout(height=500, title_text="Vehicle Telemetry", showlegend=False)
fig.show()

## 👨‍🚀 Crew Vitals

In [ ]:
crew_query = f"""
CrewVitals
| where SessionId == "{SESSION_ID}"
| summarize AvgHR=avg(HeartRate), AvgO2=avg(OxygenSaturation), 
            MaxHR=max(HeartRate), MinO2=min(OxygenSaturation) by CrewName
"""
crew_df = run_query(crew_query)

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'xy'}]],
                    subplot_titles=('O2 Saturation', 'Heart Rate by Crew'))

# O2 gauge for each crew member
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

# Heart rate bar chart
fig.add_trace(go.Bar(x=crew_df['CrewName'], y=crew_df['AvgHR'], 
                     marker_color=colors, name='Avg HR'), row=1, col=2)
fig.add_trace(go.Scatter(x=crew_df['CrewName'], y=crew_df['MaxHR'],
                         mode='markers', marker=dict(size=12, symbol='diamond'),
                         name='Max HR'), row=1, col=2)

fig.update_layout(height=400, title_text="Crew Health Status")
fig.show()

# Display crew stats table
print("\n👨‍🚀 Crew Vitals Summary:")
display(crew_df)

## 🌡️ Environmental Conditions

In [ ]:
env_query = f"""
EnvironmentalConditions
| where SessionId == "{SESSION_ID}"
| order by MissionTime asc
| project MissionTime, CabinPressure, CabinTemperature, RadiationLevel, CO2Level
"""
env_df = run_query(env_query)

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=('Cabin Pressure (kPa)', 'Temperature (°C)', 
                                   'Radiation (mSv/h)', 'CO2 (ppm)'))

fig.add_trace(go.Scatter(x=env_df['MissionTime'], y=env_df['CabinPressure'],
                         mode='lines', line=dict(color='purple')), row=1, col=1)
fig.add_trace(go.Scatter(x=env_df['MissionTime'], y=env_df['CabinTemperature'],
                         mode='lines', line=dict(color='red')), row=1, col=2)
fig.add_trace(go.Scatter(x=env_df['MissionTime'], y=env_df['RadiationLevel'],
                         mode='lines', line=dict(color='yellow')), row=2, col=1)
fig.add_trace(go.Scatter(x=env_df['MissionTime'], y=env_df['CO2Level'],
                         mode='lines', line=dict(color='gray')), row=2, col=2)

fig.update_layout(height=500, title_text="Environmental Conditions", showlegend=False)
fig.show()

## 📋 Mission Timeline

In [ ]:
events_query = f"""
MissionEvents
| where SessionId == "{SESSION_ID}"
| project MissionTime, EventName, Description, Phase
| order by MissionTime asc
"""
events_df = run_query(events_query)

print("📋 Mission Events Timeline:\n")
for _, row in events_df.iterrows():
    t = int(row['MissionTime'])
    mins = t // 60
    secs = t % 60
    print(f"  T+{mins:02d}:{secs:02d} | {row['EventName']}")
    print(f"          {row['Description']}")
    print()

## 🔄 Live Dashboard Mode

Run this cell during an active mission for real-time updates:

In [ ]:
# Uncomment to enable live refresh (Ctrl+C to stop)
# REFRESH_INTERVAL = 2  # seconds
# 
# while True:
#     clear_output(wait=True)
#     
#     # Get latest status
#     status_query = f"""
#     VehicleTelemetry
#     | where SessionId == "{SESSION_ID}"
#     | top 1 by MissionTime desc
#     """
#     status = run_query(status_query)
#     
#     if len(status) > 0:
#         row = status.iloc[0]
#         print(f"🚀 MISSION STATUS - T+{int(row['MissionTime'])}s")
#         print(f"   Phase: {row['Phase']}")
#         print(f"   Altitude: {row['Altitude']:,.0f} km")
#         print(f"   Velocity: {row['Velocity']:,.0f} km/h")
#         print(f"   Fuel: {row['FuelRemaining']:.1f}%")
#     
#     time.sleep(REFRESH_INTERVAL)